In [1]:
import pandas as pd
import numpy as np

def read_excel(file_name):
    df = pd.read_excel(file_name)
    return df

def read_txt(file_name):
    file = open(file_name)
    lines = file.readlines()
    return(lines[0])

In [2]:
import os
import glob

def get_files(subfolder, extension):
    dir = f"{os.getcwd()}/content/{subfolder}/"
    tables = glob.glob(f"{dir}*.{extension}")
    return tables

In [3]:
class Analizer:
    def __init__(self, boundary):
        self.results = get_files(subfolder="results", extension="xlsx")
        self.results_df = pd.DataFrame()
        self.boundary = boundary
    
    def has_minimum_requirements(self, df, sort_by="r2"):
        sorted_df = df.sort_values(by=sort_by, ascending=False)
        top_r2 = sorted_df.head(1)[sort_by].values[0]
        if top_r2 < self.boundary:
            return False
        return True
    
    def concatenate_df(self, df, architecture):
        if self.has_minimum_requirements(df):
            df['Architecture'] = architecture
            df = df.rename(columns={'Unnamed: 0': 'model'})
            self.results_df = pd.concat([self.results_df, df], ignore_index=True) 

    def create_results_df(self):
        for file in self.results:
            df = read_excel(file)
            architecture = read_txt(file.replace(".xlsx", ".txt"))
            self.concatenate_df(df, architecture)
        self.results_df = self.results_df.sort_values(by="r2", ascending=False, ignore_index=True)

    def discard_below_average(self, sort_by):
        column_mean = self.results_df[sort_by].mean()      
        self.results_df = self.results_df[self.results_df[sort_by] >= column_mean]
    
    def discard_high_standard_deviation(self):
        r2_val, r2_test = self.results_df['r2_val'], self.results_df['r2_test']
        std_devs = np.abs(r2_val - r2_test)
        mean_std_dev = std_devs.mean()
        self.results_df = self.results_df[std_devs < mean_std_dev]

    def clean_folder(self, subfolder, extension, remove_last=True):
        files = get_files(subfolder, extension)
        models = self.results_df["model"]
        if (remove_last):
            models = models.apply(lambda x: '_'.join(x.rsplit('_', 1)[:-1]))
        for file in files:
            file_name = os.path.basename(file).split('.')[0]
            file_parts = file_name.split('_')            
            dataset_model = f"model_{file_parts[1]}_{file_parts[2]}" 
            if (remove_last == False):
                dataset_model = (f"{dataset_model}_{file_parts[3]}")
            if dataset_model not in models.values:
                os.remove(file)   
        
    def Analize(self):
        self.create_results_df()
        self.discard_below_average(sort_by="r2_sup")
        self.discard_below_average(sort_by="r2_vt")
        self.discard_high_standard_deviation()
        self.results_df.to_excel(f"better_results.xlsx", index=True)
        display(self.results_df)


In [4]:
analize = Analizer(0.9)
analize.Analize()
analize.clean_folder(subfolder="dataset", extension="pkl")
analize.clean_folder(subfolder="results", extension="xlsx")
analize.clean_folder(subfolder="results", extension="txt")
analize.clean_folder(subfolder="models", extension="keras", remove_last=False)



,model,r2,r2_sup,r2_test,r2_val,r2_vt,mse,mse_sup,mse_test,mse_val,mse_vt,mape,rmse,r2_adj,rsd,aic,bic,Architecture
0,model_20_8_2,0.996385,0.822630,0.997405,0.991818,0.996196,0.024171,1.186073,0.027360,0.029501,0.028430,0.112986,0.155472,1.001637,0.162091,161.445165,255.298603,"Hidden Size=[19], regularizer=0.05, learning_r..."
3,model_20_8_3,0.996300,0.821566,0.997108,0.986751,0.994765,0.024743,1.193190,0.030489,0.047769,0.039129,0.122434,0.157301,1.001676,0.163997,161.398389,255.251828,"Hidden Size=[19], regularizer=0.05, learning_r..."
5,model_20_8_1,0.996285,0.823639,0.997684,0.995929,0.997385,0.024843,1.179329,0.024414,0.014678,0.019546,0.103211,0.157618,1.001682,0.164328,161.390327,255.243766,"Hidden Size=[19], regularizer=0.05, learning_r..."
15,model_20_8_0,0.995936,0.824549,0.997935,0.998658,0.998220,0.027179,1.173240,0.021777,0.004837,0.013307,0.092694,0.164859,1.001840,0.171878,161.210655,255.064094,"Hidden Size=[19], regularizer=0.05, learning_r..."
56,model_4_8_4,0.993515,0.820663,0.997146,0.982441,0.985827,0.043363,1.199230,0.003997,0.082866,0.043431,0.304419,0.208238,1.004206,0.217103,128.276300,202.627725,"Hidden Size=[15], regularizer=0.05, learning_r..."
100,model_34_6_8,0.980691,0.836746,0.962938,0.963701,0.963594,0.129120,1.091679,0.164853,0.560728,0.362791,0.082295,0.359333,1.006716,0.374631,190.094020,303.449472,"Hidden Size=[23], regularizer=0.2, learning_ra..."
101,model_34_6_7,0.980666,0.837959,0.962535,0.964986,0.964500,0.129286,1.083569,0.166643,0.540878,0.353760,0.082768,0.359563,1.006725,0.374871,190.091461,303.446913,"Hidden Size=[23], regularizer=0.2, learning_ra..."
102,model_34_6_9,0.980637,0.835571,0.963198,0.962479,0.962705,0.129480,1.099540,0.163694,0.579600,0.371647,0.081881,0.359834,1.006735,0.375153,190.088453,303.443904,"Hidden Size=[23], regularizer=0.2, learning_ra..."
103,model_34_6_6,0.980535,0.839190,0.961951,0.966325,0.965408,0.130164,1.075336,0.169244,0.520186,0.344715,0.083312,0.360782,1.006771,0.376142,190.077922,303.433374,"Hidden Size=[23], regularizer=0.2, learning_ra..."
104,model_34_6_10,0.980527,0.834445,0.963349,0.961327,0.961845,0.130218,1.107068,0.163023,0.597405,0.380214,0.081521,0.360857,1.006773,0.376219,190.077096,303.432547,"Hidden Size=[23], regularizer=0.2, learning_ra..."
